In [1]:
# Load the SQL toolset extension
%load_ext sql

In [2]:
#2. Connect to (and create) the database file
%sql sqlite:///wk6_joins_intro_jupyter.db

Connecting to 'sqlite:///wk6_joins_intro_jupyter.db'

### 1. Create the Customers table first (Parent table) 

In [3]:
%%sql
CREATE TABLE Customers (
    CustomerID INTEGER PRIMARY KEY,
    FirstName TEXT NOT NULL,
    LastName TEXT,
    Country TEXT,
    Score INTEGER
);

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

++
||
++
++

### 2. Create the Orders table (Child table with Foreign Key)

In [4]:
%%sql
CREATE TABLE Orders (
    OrderID INTEGER PRIMARY KEY,
    ProductID INTEGER,
    CustomerID INTEGER,
    SalesPersonID INTEGER,
    OrderDate TEXT,
    ShipDate TEXT,
    OrderStatus TEXT,
    ShipAddress TEXT,
    BillAddress TEXT,
    Quantity INTEGER,
    Sales REAL,
    CreationTime TEXT,
    FOREIGN KEY (CustomerID) REFERENCES Customers(CustomerID)
);

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

++
||
++
++

### 3. Import .csv files

In [5]:
import pandas as pd

# 1. Read the CSVs into memory
df_customers = pd.read_csv('Customers.csv')
df_orders = pd.read_csv('Orders.csv')

In [28]:
# Increase to 100 rows
pd.set_option('display.max_rows', 100)
%config SqlMagic.displaylimit = 50

In [6]:
print(df_customers.columns)
print(df_orders.columns)

Index(['CustomerID', 'FirstName', 'LastName', 'Country', 'Score'], dtype='str')
Index(['OrderID', 'ProductID', 'CustomerID', 'SalesPersonID', 'OrderDate',
       'ShipDate', 'OrderStatus', 'ShipAddress', 'BillAddress', 'Quantity',
       'Sales', 'CreationTime'],
      dtype='str')


In [7]:
# Step 1: Push the CSVs as staging tables
# Use the connection we already established with %sql
# Use the --no-index flag to prevent the index from being created
%sql --persist --no-index df_customers
%sql --persist --no-index df_orders

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

ValueError: Table 'df_customers' already exists. Consider using --persist-replace to drop the table before persisting the data frame


### Step 2: Use SQL to Append the data
Now, run a SQL query to move that data into your pre-existing tables (Customers and Orders). This preserves your original schema constraints.

In [8]:
%%sql
-- Move data from staging to the real table
INSERT INTO Customers 
SELECT * FROM df_customers;

-- Move data from staging to the real Orders table
INSERT INTO Orders 
SELECT * FROM df_orders;

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

5 rows affected.

10 rows affected.

++
||
++
++

In [9]:
# Verify ingestion
%sql SELECT * FROM orders LIMIT 5;

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

OrderID,ProductID,CustomerID,SalesPersonID,OrderDate,ShipDate,OrderStatus,ShipAddress,BillAddress,Quantity,Sales,CreationTime
1,101,2,3,2025-01-01,2025-01-05,Delivered,9833 Mt. Dias Blv.,1226 Shoe St.,1,10.0,2025-01-01 12:34:56.0000000
2,102,3,3,2025-01-05,2025-01-10,Shipped,250 Race Court,None,1,15.0,2025-01-05 23:22:04.0000000
3,101,1,5,2025-01-10,2025-01-25,Delivered,8157 W. Book,8157 W. Book,2,20.0,2025-01-10 18:24:08.0000000
4,105,1,3,2025-01-20,2025-01-25,Shipped,5724 Victory Lane,None,2,60.0,2025-01-20 05:50:33.0000000
5,104,2,5,2025-02-01,2025-02-05,Delivered,None,None,1,25.0,2025-02-01 14:02:41.0000000


In [10]:
%%sql
-- Remove the temporary staging tables
DROP TABLE IF EXISTS df_customers;
DROP TABLE IF EXISTS df_orders;

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

++
||
++
++

In [12]:
%%sql
-- Confirm that both tables were deleted
SELECT name FROM sqlite_master WHERE type='table';

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

name
Customers
Orders


# Run the Queries

### **1. NO JOIN:** Get all the data without joining tables
**Task:** Retrieve all data from customers and orders as separate results

In [16]:
# Display the 'customers' table
%sql SELECT * FROM customers;

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

CustomerID,FirstName,LastName,Country,Score
1,Jossef,Goldberg,Germany,350
2,Kevin,Brown,USA,900
3,Mary,None,USA,750
4,Mark,Schwarz,Germany,500
5,Anna,Adams,USA,None


In [29]:
# Display the 'orders' table
%sql SELECT * FROM orders;

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

OrderID,ProductID,CustomerID,SalesPersonID,OrderDate,ShipDate,OrderStatus,ShipAddress,BillAddress,Quantity,Sales,CreationTime
1,101,2,3,2025-01-01,2025-01-05,Delivered,9833 Mt. Dias Blv.,1226 Shoe St.,1,10.0,2025-01-01 12:34:56.0000000
2,102,3,3,2025-01-05,2025-01-10,Shipped,250 Race Court,None,1,15.0,2025-01-05 23:22:04.0000000
3,101,1,5,2025-01-10,2025-01-25,Delivered,8157 W. Book,8157 W. Book,2,20.0,2025-01-10 18:24:08.0000000
4,105,1,3,2025-01-20,2025-01-25,Shipped,5724 Victory Lane,None,2,60.0,2025-01-20 05:50:33.0000000
5,104,2,5,2025-02-01,2025-02-05,Delivered,None,None,1,25.0,2025-02-01 14:02:41.0000000
6,104,3,5,2025-02-05,2025-02-10,Delivered,1792 Belmont Rd.,None,2,50.0,2025-02-06 15:34:57.0000000
7,102,1,1,2025-02-15,2025-02-27,Delivered,136 Balboa Court,None,2,30.0,2025-02-16 06:22:01.0000000
8,101,4,3,2025-02-18,2025-02-27,Shipped,2947 Vine Lane,4311 Clay Rd,3,90.0,2025-02-18 10:45:22.0000000
9,101,2,3,2025-03-10,2025-03-15,Shipped,3768 Door Way,None,2,20.0,2025-03-10 12:59:04.0000000
10,102,3,5,2025-03-15,2025-03-20,Shipped,None,None,0,60.0,2025-03-16 23:25:15.0000000


### **2. Inner JOIN:** Returns only the matching rows from both tables. We only want the matching data from both tables.
**Task:** Get all customers along with their orders, but only for customers who have placed an order. 

In [30]:
%%sql
-- No aliases, retrieving all columns
SELECT *
FROM customers
INNER JOIN orders
ON customers.CustomerID = orders.CustomerID

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

CustomerID,FirstName,LastName,Country,Score,OrderID,ProductID,CustomerID_1,SalesPersonID,OrderDate,ShipDate,OrderStatus,ShipAddress,BillAddress,Quantity,Sales,CreationTime
2,Kevin,Brown,USA,900,1,101,2,3,2025-01-01,2025-01-05,Delivered,9833 Mt. Dias Blv.,1226 Shoe St.,1,10.0,2025-01-01 12:34:56.0000000
3,Mary,None,USA,750,2,102,3,3,2025-01-05,2025-01-10,Shipped,250 Race Court,None,1,15.0,2025-01-05 23:22:04.0000000
1,Jossef,Goldberg,Germany,350,3,101,1,5,2025-01-10,2025-01-25,Delivered,8157 W. Book,8157 W. Book,2,20.0,2025-01-10 18:24:08.0000000
1,Jossef,Goldberg,Germany,350,4,105,1,3,2025-01-20,2025-01-25,Shipped,5724 Victory Lane,None,2,60.0,2025-01-20 05:50:33.0000000
2,Kevin,Brown,USA,900,5,104,2,5,2025-02-01,2025-02-05,Delivered,None,None,1,25.0,2025-02-01 14:02:41.0000000
3,Mary,None,USA,750,6,104,3,5,2025-02-05,2025-02-10,Delivered,1792 Belmont Rd.,None,2,50.0,2025-02-06 15:34:57.0000000
1,Jossef,Goldberg,Germany,350,7,102,1,1,2025-02-15,2025-02-27,Delivered,136 Balboa Court,None,2,30.0,2025-02-16 06:22:01.0000000
4,Mark,Schwarz,Germany,500,8,101,4,3,2025-02-18,2025-02-27,Shipped,2947 Vine Lane,4311 Clay Rd,3,90.0,2025-02-18 10:45:22.0000000
2,Kevin,Brown,USA,900,9,101,2,3,2025-03-10,2025-03-15,Shipped,3768 Door Way,None,2,20.0,2025-03-10 12:59:04.0000000
3,Mary,None,USA,750,10,102,3,5,2025-03-15,2025-03-20,Shipped,None,None,0,60.0,2025-03-16 23:25:15.0000000


In [32]:
%%sql
-- select columns that only make sense in our query
SELECT
    customers.CustomerID,
    FirstName,
    LastName,
    Sales
FROM customers
JOIN orders
ON customers.CustomerID = orders.CustomerID

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

CustomerID,FirstName,LastName,Sales
2,Kevin,Brown,10.0
3,Mary,None,15.0
1,Jossef,Goldberg,20.0
1,Jossef,Goldberg,60.0
2,Kevin,Brown,25.0
3,Mary,None,50.0
1,Jossef,Goldberg,30.0
4,Mark,Schwarz,90.0
2,Kevin,Brown,20.0
3,Mary,None,60.0


In [33]:
%%sql
-- Column ambiguity (best practices)
SELECT
    customers.CustomerID,
    customers.FirstName,
    customers.LastName,
    orders.Sales
FROM customers
JOIN orders
ON customers.CustomerID = orders.CustomerID

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

CustomerID,FirstName,LastName,Sales
2,Kevin,Brown,10.0
3,Mary,None,15.0
1,Jossef,Goldberg,20.0
1,Jossef,Goldberg,60.0
2,Kevin,Brown,25.0
3,Mary,None,50.0
1,Jossef,Goldberg,30.0
4,Mark,Schwarz,90.0
2,Kevin,Brown,20.0
3,Mary,None,60.0


In [36]:
%%sql
-- Aliases
SELECT
    c.CustomerID,
    c.FirstName,
    c.LastName,
    o.Sales
FROM customers AS c
JOIN orders AS o
ON c.CustomerID = o.CustomerID

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

CustomerID,FirstName,LastName,Sales
2,Kevin,Brown,10.0
3,Mary,None,15.0
1,Jossef,Goldberg,20.0
1,Jossef,Goldberg,60.0
2,Kevin,Brown,25.0
3,Mary,None,50.0
1,Jossef,Goldberg,30.0
4,Mark,Schwarz,90.0
2,Kevin,Brown,20.0
3,Mary,None,60.0


In [38]:
%%sql
-- Switch tables (same results)
SELECT
    c.CustomerID,
    c.FirstName,
    c.LastName,
    o.Sales
FROM orders AS o
JOIN customers AS c
ON c.CustomerID = o.CustomerID

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

CustomerID,FirstName,LastName,Sales
2,Kevin,Brown,10.0
3,Mary,None,15.0
1,Jossef,Goldberg,20.0
1,Jossef,Goldberg,60.0
2,Kevin,Brown,25.0
3,Mary,None,50.0
1,Jossef,Goldberg,30.0
4,Mark,Schwarz,90.0
2,Kevin,Brown,20.0
3,Mary,None,60.0


### **3. Left JOIN:** Returns all rows from left and only matching from right.
**Task:** et all customers along with their orders, including those without orders. 

In [40]:
%%sql
SELECT
    c.CustomerID,
    c.FirstName,
    o.OrderID,
    o.sales
FROM customers AS c
LEFT JOIN orders AS o
ON c.CustomerID = o.CustomerID

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

CustomerID,FirstName,OrderID,Sales
1,Jossef,3,20.0
1,Jossef,7,30.0
1,Jossef,4,60.0
2,Kevin,1,10.0
2,Kevin,9,20.0
2,Kevin,5,25.0
3,Mary,2,15.0
3,Mary,6,50.0
3,Mary,10,60.0
4,Mark,8,90.0


In [41]:
%%sql
-- Switch the order of the tables
SELECT
    c.CustomerID,
    c.FirstName,
    o.OrderID,
    o.sales
FROM orders AS o 
LEFT JOIN customers AS c
ON c.CustomerID = o.CustomerID

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

CustomerID,FirstName,OrderID,Sales
2,Kevin,1,10.0
3,Mary,2,15.0
1,Jossef,3,20.0
1,Jossef,4,60.0
2,Kevin,5,25.0
3,Mary,6,50.0
1,Jossef,7,30.0
4,Mark,8,90.0
2,Kevin,9,20.0
3,Mary,10,60.0


### **4. Right JOIN:** Returns all rows from right and only matching from left.
**Task:** Get all customers along with their orders, including orders without matching customers. 

In [44]:
%%sql
SELECT
    c.CustomerID,
    c.FirstName,
    o.OrderID,
    o.sales
FROM customers AS c 
RIGHT JOIN orders AS o
ON c.CustomerID = o.CustomerID

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

CustomerID,FirstName,OrderID,Sales
1,Jossef,3,20.0
1,Jossef,4,60.0
1,Jossef,7,30.0
2,Kevin,1,10.0
2,Kevin,5,25.0
2,Kevin,9,20.0
3,Mary,2,15.0
3,Mary,6,50.0
3,Mary,10,60.0
4,Mark,8,90.0


In [45]:
%%sql
-- Alternative to RIGHT JOIN using LEFT JOIN
SELECT
    c.CustomerID,
    c.FirstName,
    o.OrderID,
    o.sales
FROM orders AS o 
LEFT JOIN customers AS c
ON c.CustomerID = o.CustomerID

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

CustomerID,FirstName,OrderID,Sales
2,Kevin,1,10.0
3,Mary,2,15.0
1,Jossef,3,20.0
1,Jossef,4,60.0
2,Kevin,5,25.0
3,Mary,6,50.0
1,Jossef,7,30.0
4,Mark,8,90.0
2,Kevin,9,20.0
3,Mary,10,60.0


### **5. Full JOIN:** Returns all rows from both tables.
**Task:** Get all customers and all orders, even if there’s no match. 

In [48]:
%%sql
SELECT
    c.CustomerID,
    c.FirstName,
    c.LastName,
    o.OrderID,
    -- o.CustomerID,
    o.Sales
FROM orders AS o 
FULL JOIN customers AS c
ON c.CustomerID = o.CustomerID

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

CustomerID,FirstName,LastName,OrderID,Sales
2,Kevin,Brown,1,10.0
3,Mary,None,2,15.0
1,Jossef,Goldberg,3,20.0
1,Jossef,Goldberg,4,60.0
2,Kevin,Brown,5,25.0
3,Mary,None,6,50.0
1,Jossef,Goldberg,7,30.0
4,Mark,Schwarz,8,90.0
2,Kevin,Brown,9,20.0
3,Mary,None,10,60.0


### **6. Cross JOIN:** Combines every row from left with every row from the right. All possible combinations - Cartesian Join
**Task:** Generate all possible combinations of customers and orders. 

In [49]:
%%sql
SELECT *
FROM customers
CROSS JOIN orders

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

CustomerID,FirstName,LastName,Country,Score,OrderID,ProductID,CustomerID_1,SalesPersonID,OrderDate,ShipDate,OrderStatus,ShipAddress,BillAddress,Quantity,Sales,CreationTime
1,Jossef,Goldberg,Germany,350,1,101,2,3,2025-01-01,2025-01-05,Delivered,9833 Mt. Dias Blv.,1226 Shoe St.,1,10.0,2025-01-01 12:34:56.0000000
1,Jossef,Goldberg,Germany,350,2,102,3,3,2025-01-05,2025-01-10,Shipped,250 Race Court,None,1,15.0,2025-01-05 23:22:04.0000000
1,Jossef,Goldberg,Germany,350,3,101,1,5,2025-01-10,2025-01-25,Delivered,8157 W. Book,8157 W. Book,2,20.0,2025-01-10 18:24:08.0000000
1,Jossef,Goldberg,Germany,350,4,105,1,3,2025-01-20,2025-01-25,Shipped,5724 Victory Lane,None,2,60.0,2025-01-20 05:50:33.0000000
1,Jossef,Goldberg,Germany,350,5,104,2,5,2025-02-01,2025-02-05,Delivered,None,None,1,25.0,2025-02-01 14:02:41.0000000
1,Jossef,Goldberg,Germany,350,6,104,3,5,2025-02-05,2025-02-10,Delivered,1792 Belmont Rd.,None,2,50.0,2025-02-06 15:34:57.0000000
1,Jossef,Goldberg,Germany,350,7,102,1,1,2025-02-15,2025-02-27,Delivered,136 Balboa Court,None,2,30.0,2025-02-16 06:22:01.0000000
1,Jossef,Goldberg,Germany,350,8,101,4,3,2025-02-18,2025-02-27,Shipped,2947 Vine Lane,4311 Clay Rd,3,90.0,2025-02-18 10:45:22.0000000
1,Jossef,Goldberg,Germany,350,9,101,2,3,2025-03-10,2025-03-15,Shipped,3768 Door Way,None,2,20.0,2025-03-10 12:59:04.0000000
1,Jossef,Goldberg,Germany,350,10,102,3,5,2025-03-15,2025-03-20,Shipped,None,None,0,60.0,2025-03-16 23:25:15.0000000


#### **7. Multiple JOIN:** 
**Task:** Retrieve a list of all orders, along with the related customer, product, and employee details. For each order, display:
   - Order ID
   - Customer's name
   - Product name
   - Sales amount
   - Product price
   - Salesperson's name 

In [60]:
%%sql
-- Create the table Employees
CREATE TABLE Employees (
    EmployeeID INTEGER PRIMARY KEY,
    FirstName TEXT NOT NULL,
    LastName TEXT,
    Department TEXT,
    BirthDate TEXT, -- SQLite uses TEXT for dates
    Gender TEXT,
    Salary INTEGER,
    ManagerID INTEGER,
    FOREIGN KEY (ManagerID) REFERENCES Employees(EmployeeID)
);

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

RuntimeError: (sqlite3.OperationalError) table Employees already exists
[SQL: CREATE TABLE Employees (
    EmployeeID INTEGER PRIMARY KEY,
    FirstName TEXT NOT NULL,
    LastName TEXT,
    Department TEXT,
    BirthDate TEXT,
    Gender TEXT,
    Salary INTEGER,
    ManagerID INTEGER,
    FOREIGN KEY (ManagerID) REFERENCES Employees(EmployeeID)
);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [61]:
%%sql
-- Create the table Products
CREATE TABLE Products (
    ProductID INTEGER PRIMARY KEY,
    Product TEXT NOT NULL,
    Category TEXT,
    Price REAL
);

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

RuntimeError: (sqlite3.OperationalError) table Products already exists
[SQL: CREATE TABLE Products (
    ProductID INTEGER PRIMARY KEY,
    Product TEXT NOT NULL,
    Category TEXT,
    Price REAL
);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [50]:
# 1. Read the CSVs into memory
df_products = pd.read_csv('Products.csv')
df_employees = pd.read_csv('Employees.csv')

In [51]:
print(df_products.columns)
print(df_employees.columns)

Index(['ProductID', 'Product', 'Category', 'Price'], dtype='str')
Index(['EmployeeID', 'FirstName', 'LastName', 'Department', 'BirthDate',
       'Gender', 'Salary', 'ManagerID'],
      dtype='str')


In [52]:
# Step 1: Push the CSVs as staging tables
# Use the connection we already established with %sql
# Use the --no-index flag to prevent the index from being created
%sql --persist --no-index df_products
%sql --persist --no-index df_employees

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

Success! Persisted df_products to the database.

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

Success! Persisted df_employees to the database.

In [55]:
%%sql
-- Move data from staging to the real Employees table
INSERT INTO Employees 
SELECT * FROM df_employees;

-- Move data from staging to the real Products table
INSERT INTO Products 
SELECT * FROM df_products;

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

5 rows affected.

5 rows affected.

++
||
++
++

In [63]:
# Verify ingestion
%sql SELECT * FROM Products LIMIT 5;

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

ProductID,Product,Category,Price
101,Bottle,Accessories,10.0
102,Tire,Accessories,15.0
103,Socks,Clothing,20.0
104,Caps,Clothing,25.0
105,Gloves,Clothing,30.0


In [64]:
%%sql
-- Remove the temporary staging tables
DROP TABLE IF EXISTS df_employees;
DROP TABLE IF EXISTS df_products;

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

++
||
++
++

In [65]:
# Confirm that both tables were deleted
%sql SELECT name FROM sqlite_master WHERE type='table';

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

name
Customers
Orders
Products
Employees


**Task:** Retrieve a list of all orders, along with the related customer, product, and employee details. For each order, display:
   - Order ID
   - Customer's name
   - Product name
   - Sales amount
   - Product price
   - Salesperson's name 

In [75]:
%%sql
SELECT 
    o.OrderID,
    o.Sales,
    c.FirstName AS CustomerFirstName,
    c.LastName AS CustomerLastName,
    p.Product AS ProductName,
    p.Price,
    e.FirstName AS EmployeeFirstName,
    e.LastName AS EmployeeLastName
FROM Orders AS o
LEFT JOIN Customers AS c
ON o.CustomerID = c.CustomerID
LEFT JOIN Products AS p
ON o.ProductID = p.ProductID
LEFT JOIN Employees AS e
ON o.SalesPersonID = e.EmployeeID

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

OrderID,Sales,CustomerFirstName,CustomerLastName,ProductName,Price,EmployeeFirstName,EmployeeLastName
1,10.0,Kevin,Brown,Bottle,10.0,Mary,None
2,15.0,Mary,None,Tire,15.0,Mary,None
3,20.0,Jossef,Goldberg,Bottle,10.0,Carol,Baker
4,60.0,Jossef,Goldberg,Gloves,30.0,Mary,None
5,25.0,Kevin,Brown,Caps,25.0,Carol,Baker
6,50.0,Mary,None,Caps,25.0,Carol,Baker
7,30.0,Jossef,Goldberg,Tire,15.0,Frank,Lee
8,90.0,Mark,Schwarz,Bottle,10.0,Mary,None
9,20.0,Kevin,Brown,Bottle,10.0,Mary,None
10,60.0,Mary,None,Tire,15.0,Carol,Baker


In [69]:
%sql SELECT * FROM Customers

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

CustomerID,FirstName,LastName,Country,Score
1,Jossef,Goldberg,Germany,350
2,Kevin,Brown,USA,900
3,Mary,None,USA,750
4,Mark,Schwarz,Germany,500
5,Anna,Adams,USA,None


#### **8. Self JOIN:** 
**Task:** Write a query to display a list of all employees alongside the name of their direct manager.

In [80]:
%%sql
SELECT 
    e.FirstName AS Employee_Name, 
    m.FirstName AS Manager_Name  -- This is the data that doesn't exist in 'e'
FROM Employees e
LEFT JOIN Employees m ON e.ManagerID = m.EmployeeID;

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

Employee_Name,Manager_Name
Frank,None
Kevin,Frank
Mary,Frank
Michael,Kevin
Carol,Mary


In [78]:
%sql SELECT * FROM Employees

Running query in 'sqlite:///wk6_joins_intro_jupyter.db'

EmployeeID,FirstName,LastName,Department,BirthDate,Gender,Salary,ManagerID
1,Frank,Lee,Marketing,1988-12-05,M,55000,None
2,Kevin,Brown,Marketing,1972-11-25,M,65000,1
3,Mary,None,Sales,1986-01-05,F,75000,1
4,Michael,Ray,Sales,1977-02-10,M,90000,2
5,Carol,Baker,Sales,1982-02-11,F,55000,3


In [ ]:
# Close the connection
%sql --close sqlite:///wk6_joins_intro_test.db